In [30]:
import sys
sys.path.append("..")

import requests
import pandas as pd
import yfinance as yf
import time
from src.features import calcular_surpresa, calcular_ian, calcular_ice

In [31]:
url_ipca = "https://api.bcb.gov.br/dados/serie/bcdata.sgs.433/dados?formato=json"
resposta = requests.get(url_ipca)
ipca = pd.DataFrame(resposta.json())
ipca["data"] = pd.to_datetime(ipca["data"], dayfirst=True)
ipca = ipca[ipca["data"] >= "2023-01-01"]
print(f"IPCA realizado: {len(ipca)} linhas")

IPCA realizado: 43 linhas


In [32]:
url_expectativas = (
    "https://olinda.bcb.gov.br/olinda/servico/Expectativas/versao/v1/odata/"
    "ExpectativaMercadoMensais?$top=10000&$format=json"
    "&$filter=Indicador%20eq%20%27IPCA%27%20and%20Data%20ge%20%272023-01-01%27"
    "&$orderby=Data%20asc"
)
resposta = requests.get(url_expectativas)
expectativas = pd.DataFrame(resposta.json()["value"])
expectativas["Data"] = pd.to_datetime(expectativas["Data"])
expectativas["DataReferencia"] = pd.to_datetime(expectativas["DataReferencia"], format="%m/%Y")

expectativas = expectativas.sort_values("Data")
consenso_ipca = expectativas.groupby("DataReferencia").last().reset_index()
print(f"Consenso IPCA: {len(consenso_ipca)} meses")

Consenso IPCA: 35 meses


In [33]:
ipca_realizado = ipca.rename(columns={"data": "DataReferencia", "valor": "actual"})
ipca_realizado["DataReferencia"] = pd.to_datetime(ipca_realizado["DataReferencia"])
ipca_realizado["actual"] = ipca_realizado["actual"].astype(float)

eventos_ipca_bruto = ipca_realizado.merge(
    consenso_ipca[["DataReferencia", "Mediana"]], on="DataReferencia", how="inner"
)
eventos_ipca_bruto = eventos_ipca_bruto.rename(columns={"Mediana": "forecast"})
print(f"IPCA com consenso: {len(eventos_ipca_bruto)} linhas")

IPCA com consenso: 34 linhas


In [34]:
datas_ipca = pd.read_csv("../data/ipca.csv", sep=";")
datas_ipca = datas_ipca.dropna(subset=["Actual"])

datas_ipca["data_divulgacao"] = pd.to_datetime(
    datas_ipca["Release date"].str.split("(").str[0].str.strip(), format="%b %d, %Y"
)

mes_ref_texto = datas_ipca["Release date"].str.extract(r"\((\w+)\)")[0]
meses = {"Jan":1,"Feb":2,"Mar":3,"Apr":4,"May":5,"Jun":6,"Jul":7,"Aug":8,"Sep":9,"Oct":10,"Nov":11,"Dec":12}
mes_ref_num = mes_ref_texto.map(meses)

ano_divulgacao = datas_ipca["data_divulgacao"].dt.year
mes_divulgacao = datas_ipca["data_divulgacao"].dt.month
ano_ref = ano_divulgacao.where(mes_ref_num <= mes_divulgacao, ano_divulgacao - 1)

datas_ipca["DataReferencia"] = pd.to_datetime(
    ano_ref.astype(str) + "-" + mes_ref_num.astype(str) + "-01"
)
print(f"Datas de divulgação: {len(datas_ipca)} linhas")

Datas de divulgação: 43 linhas


In [35]:
eventos_ipca_bruto = eventos_ipca_bruto.merge(
    datas_ipca[["DataReferencia", "data_divulgacao"]], on="DataReferencia", how="inner"
)
eventos_ipca_bruto["indicador"] = "IPCA_BR"
eventos_ipca_bruto["data"] = eventos_ipca_bruto["data_divulgacao"].dt.strftime("%Y-%m-%d")

eventos_ipca_final = eventos_ipca_bruto[["indicador", "data", "actual", "forecast"]].copy()
print(f"IPCA final: {len(eventos_ipca_final)} linhas")
print(eventos_ipca_final.head())

IPCA final: 34 linhas
  indicador        data  actual  forecast
0   IPCA_BR  2023-02-09    0.53     0.555
1   IPCA_BR  2023-03-10    0.84     0.780
2   IPCA_BR  2023-04-11    0.71     0.770
3   IPCA_BR  2023-05-12    0.61     0.550
4   IPCA_BR  2023-06-07    0.23     0.370


In [36]:
ewz = yf.download("EWZ", start="2023-01-01", end="2026-08-06", progress=True)
ewz = ewz[["Close"]].reset_index()
ewz.columns = ["data", "close"]
ewz["data"] = ewz["data"].dt.strftime("%Y-%m-%d")
ewz["retorno_pct"] = ewz["close"].pct_change() * 100
ewz.to_csv("../data/ewz_precos.csv", index=False)
print("EWZ salvo.")

[*********************100%***********************]  1 of 1 completed

EWZ salvo.


In [37]:
usdbrl = yf.download("BRL=X", start="2023-01-01", end="2026-08-06", progress=True)
usdbrl = usdbrl[["Close"]].reset_index()
usdbrl.columns = ["data", "close"]
usdbrl["data"] = usdbrl["data"].dt.strftime("%Y-%m-%d")
usdbrl["retorno_pct"] = usdbrl["close"].pct_change() * 100
usdbrl.to_csv("../data/usdbrl_precos.csv", index=False)
print("USD/BRL salvo.")

[*********************100%***********************]  1 of 1 completed

USD/BRL salvo.


In [38]:
url_selic = (
    "https://api.bcb.gov.br/dados/serie/bcdata.sgs.432/dados"
    "?formato=json&dataInicial=01/01/2023&dataFinal=06/08/2026"
)
resposta = requests.get(url_selic)
selic = pd.DataFrame(resposta.json())
selic["data"] = pd.to_datetime(selic["data"], dayfirst=True)
print(f"Selic diária: {len(selic)} linhas")

Selic diária: 1314 linhas


In [39]:
selic = selic.sort_values("data").reset_index(drop=True)
selic["valor_anterior"] = selic["valor"].shift(1)

decisoes = selic[selic["valor"] != selic["valor_anterior"]].copy()
decisoes = decisoes.dropna(subset=["valor_anterior"])
print(f"Decisões do Copom: {len(decisoes)}")

Decisões do Copom: 18


In [40]:
anos = [2022, 2023, 2024, 2025, 2026]
partes = []
for ano in anos:
    url = (
        "https://olinda.bcb.gov.br/olinda/servico/Expectativas/versao/v1/odata/"
        "ExpectativasMercadoSelic?$top=10000&$format=json"
        f"&$filter=Data%20ge%20%27{ano}-01-01%27%20and%20Data%20le%20%27{ano}-12-31%27%20and%20baseCalculo%20eq%200"
        "&$orderby=Data%20asc"
    )
    resposta = requests.get(url)
    parte = pd.DataFrame(resposta.json()["value"])
    partes.append(parte)

expectativas_selic = pd.concat(partes, ignore_index=True)
expectativas_selic["Data"] = pd.to_datetime(expectativas_selic["Data"])
expectativas_selic["ano_reuniao"] = expectativas_selic["Reuniao"].str.extract(r"/(\d+)").astype(int)
expectativas_selic["numero_reuniao"] = expectativas_selic["Reuniao"].str.extract(r"R(\d+)").astype(int)
print(f"Expectativas Selic: {len(expectativas_selic)} linhas")

Expectativas Selic: 18480 linhas


In [41]:
resultados = []
for _, decisao in decisoes.iterrows():
    data_decisao = decisao["data"]
    candidatos = expectativas_selic[expectativas_selic["Data"] < data_decisao]
    if candidatos.empty:
        continue
    ultima_data_coleta = candidatos["Data"].max()
    candidatos_ultima_data = candidatos[candidatos["Data"] == ultima_data_coleta]
    candidatos_ultima_data = candidatos_ultima_data.sort_values(["ano_reuniao", "numero_reuniao"])
    proxima_reuniao = candidatos_ultima_data.iloc[0]
    resultados.append({
        "data": data_decisao.strftime("%Y-%m-%d"),
        "actual": decisao["valor"],
        "forecast": proxima_reuniao["Mediana"]
    })

eventos_selic_final = pd.DataFrame(resultados)
eventos_selic_final["indicador"] = "Selic_BR"
eventos_selic_final = eventos_selic_final[["indicador", "data", "actual", "forecast"]]
print(f"Selic final: {len(eventos_selic_final)} linhas")

Selic final: 18 linhas


In [42]:
eventos_principal = pd.read_csv("../data/eventos.csv")
eventos_principal = eventos_principal[~eventos_principal["indicador"].isin(["IPCA_BR", "Selic_BR"])]

eventos_atualizado = pd.concat([eventos_principal, eventos_ipca_final, eventos_selic_final], ignore_index=True)
eventos_atualizado = eventos_atualizado.drop_duplicates(subset=["indicador", "data"])
eventos_atualizado.to_csv("../data/eventos.csv", index=False)

print(eventos_atualizado["indicador"].value_counts())

indicador
Payroll_EUA    43
CPI_EUA        39
IPCA_BR        34
Selic_BR       18
Name: count, dtype: int64


In [43]:
ipca_completo = eventos_ipca_final.copy()
print(f"Antes de calcular: {len(ipca_completo)} linhas, indicadores únicos: {ipca_completo['indicador'].unique()}")

ipca_completo = calcular_surpresa(ipca_completo)
ipca_completo = calcular_ian(ipca_completo, termos_busca=["IPCA", "inflação"], geo="BR")
ipca_completo = calcular_ice(
    ipca_completo,
    termos_otimistas=["abrir empresa", "comprar carro", "promoção passagens"],
    termos_pessimistas=["perder emprego", "inflação alta", "dívida"],
    geo="BR"
)
ipca_completo.to_csv("../data/eventos_ipca_completo.csv", index=False)
print(f"IPCA processado: {len(ipca_completo)} linhas, maior surpresa: {ipca_completo['surpresa_zscore'].abs().max():.2f}")

Antes de calcular: 34 linhas, indicadores únicos: <StringArray>
['IPCA_BR']
Length: 1, dtype: str
IPCA processado: 34 linhas, maior surpresa: 4.18


In [44]:
selic_completo = eventos_selic_final.copy()
print(f"Antes de calcular: {len(selic_completo)} linhas, indicadores únicos: {selic_completo['indicador'].unique()}")

selic_completo["actual"] = pd.to_numeric(selic_completo["actual"], errors="coerce")
selic_completo["forecast"] = pd.to_numeric(selic_completo["forecast"], errors="coerce")

selic_completo = calcular_surpresa(selic_completo)
selic_completo = calcular_ian(selic_completo, termos_busca=["Selic", "juros"], geo="BR")
selic_completo = calcular_ice(
    selic_completo,
    termos_otimistas=["abrir empresa", "comprar carro", "promoção passagens"],
    termos_pessimistas=["perder emprego", "inflação alta", "dívida"],
    geo="BR"
)
selic_completo.to_csv("../data/eventos_selic_completo.csv", index=False)
print(f"Selic processada: {len(selic_completo)} linhas, maior surpresa: {selic_completo['surpresa_zscore'].abs().max():.2f}")

Antes de calcular: 18 linhas, indicadores únicos: <StringArray>
['Selic_BR']
Length: 1, dtype: str
Selic processada: 18 linhas, maior surpresa: 2.40


In [45]:
cpi = pd.read_csv("../data/eventos_completo.csv")
payroll = pd.read_csv("../data/eventos_payroll_completo.csv")

eventos_todos = pd.concat([cpi, ipca_completo, selic_completo, payroll], ignore_index=True)
eventos_todos = eventos_todos.drop_duplicates(subset=["indicador", "data"])
eventos_todos.to_csv("../data/eventos_todos_completo.csv", index=False)

print(eventos_todos["indicador"].value_counts())
print(f"\nMaior surpresa por indicador:")
print(eventos_todos.groupby("indicador")["surpresa_zscore"].apply(lambda x: x.abs().max()))

indicador
Payroll_EUA    43
CPI_EUA        39
IPCA_BR        34
Selic_BR       18
Name: count, dtype: int64

Maior surpresa por indicador:
indicador
CPI_EUA        3.022518
IPCA_BR        4.182419
Payroll_EUA    3.864244
Selic_BR       2.402829
Name: surpresa_zscore, dtype: float64


In [46]:
datas_ipca = datas_ipca.dropna(subset=["Actual"])

datas_ipca["data_divulgacao"] = pd.to_datetime(
    datas_ipca["Release date"].str.split("(").str[0].str.strip(),
    format="%b %d, %Y"
)

mes_ref_texto = datas_ipca["Release date"].str.extract(r"\((\w+)\)")[0]
meses = {"Jan":1,"Feb":2,"Mar":3,"Apr":4,"May":5,"Jun":6,"Jul":7,"Aug":8,"Sep":9,"Oct":10,"Nov":11,"Dec":12}
mes_ref_num = mes_ref_texto.map(meses)

ano_divulgacao = datas_ipca["data_divulgacao"].dt.year
mes_divulgacao = datas_ipca["data_divulgacao"].dt.month

ano_ref = ano_divulgacao.where(mes_ref_num <= mes_divulgacao, ano_divulgacao - 1)

datas_ipca["DataReferencia"] = pd.to_datetime(
    ano_ref.astype(str) + "-" + mes_ref_num.astype(str) + "-01"
)

print(datas_ipca[["Release date", "DataReferencia", "data_divulgacao"]])

          Release date DataReferencia data_divulgacao
1   Jul 10, 2026 (Jun)     2026-06-01      2026-07-10
2   Jun 12, 2026 (May)     2026-05-01      2026-06-12
3   May 12, 2026 (Apr)     2026-04-01      2026-05-12
4   Apr 10, 2026 (Mar)     2026-03-01      2026-04-10
5   Mar 12, 2026 (Feb)     2026-02-01      2026-03-12
6   Feb 10, 2026 (Jan)     2026-01-01      2026-02-10
7   Jan 09, 2026 (Dec)     2025-12-01      2026-01-09
8   Dec 10, 2025 (Nov)     2025-11-01      2025-12-10
9   Nov 11, 2025 (Oct)     2025-10-01      2025-11-11
10  Oct 09, 2025 (Sep)     2025-09-01      2025-10-09
11  Sep 10, 2025 (Aug)     2025-08-01      2025-09-10
12  Aug 12, 2025 (Jul)     2025-07-01      2025-08-12
13  Jul 10, 2025 (Jun)     2025-06-01      2025-07-10
14  Jun 10, 2025 (May)     2025-05-01      2025-06-10
15  May 09, 2025 (Apr)     2025-04-01      2025-05-09
16  Apr 11, 2025 (Mar)     2025-03-01      2025-04-11
17  Mar 12, 2025 (Feb)     2025-02-01      2025-03-12
18  Feb 11, 2025 (Jan)     2

In [47]:
print(eventos_todos["indicador"].value_counts())
print(eventos_todos.groupby("indicador")["surpresa_zscore"].apply(lambda x: x.abs().max()))

indicador
Payroll_EUA    43
CPI_EUA        39
IPCA_BR        34
Selic_BR       18
Name: count, dtype: int64
indicador
CPI_EUA        3.022518
IPCA_BR        4.182419
Payroll_EUA    3.864244
Selic_BR       2.402829
Name: surpresa_zscore, dtype: float64
